# Directional Structure in Geopolitical Asset Embeddings

This outcome-free experiment tests whether 2021Q4 holdings embeddings contain a beneficiary-versus-adverse Ukraine exposure direction. It deliberately does not load post-invasion returns. The primary model is the transparent positive-minus-negative centroid direction; pairwise ranking is reserved for later work.

In [ ]:
from pathlib import Path
import importlib,sys,time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if ROOT.name=='geopolitical_exposure': GEO,ROOT=ROOT,ROOT.parent
else: GEO=ROOT/'geopolitical_exposure'
sys.path.insert(0,str(GEO))
RESULTS=GEO/'directional_centroid';RESULTS.mkdir(parents=True,exist_ok=True)
import src.directional_centroid as dc
importlib.reload(dc)
SEED_PATH=GEO/'embedding_expansion_v2/directional_firm_channel_seeds.parquet'
assert SEED_PATH.exists()
print('Event returns loaded: False')

## 1. Freeze the design

The primary specification is All-S12 RS-Binary 10D, equal-weighted class centroids, leave-one-firm-out validation, and 5,000 label permutations. Mixed firms are excluded from training. Other representations and dimensions are reported without selecting the best one.

In [ ]:
design=dc.write_design(RESULTS/'design.json',SEED_PATH,repetitions=5000)
display(pd.DataFrame([design]))

## 2. Construct audited firm-level labels

Positive-only firms have at least one positive accepted mechanism and no negative mechanism. Negative-only firms satisfy the reverse condition. Firms containing both signs are retained as mixed diagnostics and excluded from primary estimation.

In [ ]:
seeds=pd.read_parquet(SEED_PATH)
labels,channel_audit=dc.build_firm_labels(seeds)
labels.to_parquet(RESULTS/'firm_level_labels.parquet',index=False)
channel_audit.to_parquet(RESULTS/'channel_label_audit.parquet',index=False)
display(labels.label_state.value_counts().rename_axis('label_state').reset_index(name='firms'))
display(channel_audit)

## 3. Load pre-event representations and audit coverage

All representations must cover the identical labeled-firm sample. No representation-specific filtering is permitted.

In [ ]:
representations=dc.load_representations(ROOT)
coverage=[]
for name,frame in representations.items():
    coverage.append({'representation':name,'universe_firms':len(frame),'labeled_firms':labels.firm_id.isin(frame.permco).sum(),'primary_nonmixed_firms':labels.loc[labels.class_label.ne(0),'firm_id'].isin(frame.permco).sum()})
coverage=pd.DataFrame(coverage);coverage.to_parquet(RESULTS/'representation_coverage.parquet',index=False)
display(coverage)
assert coverage.primary_nonmixed_firms.nunique()==1 and coverage.primary_nonmixed_firms.iloc[0]==84

## 4. Primary leave-one-out centroid direction

For each labeled firm, the positive and negative centroids are recomputed without that firm. The score is its projection on the normalized centroid difference minus the training-centroid midpoint. AUC is primary; balanced accuracy uses the training-derived midpoint.

In [ ]:
metric_frames=[];prediction_frames=[]
for name,frame in representations.items():
    metrics,predictions=dc.evaluate_representation(frame,labels,name)
    metric_frames.append(metrics);prediction_frames.append(predictions)
metrics=pd.concat(metric_frames,ignore_index=True);predictions=pd.concat(prediction_frames,ignore_index=True)
metrics.to_parquet(RESULTS/'loo_metrics_all_dimensions.parquet',index=False)
predictions.to_parquet(RESULTS/'loo_predictions_all_dimensions.parquet',index=False)
primary=metrics.query("representation=='all_s12_rs_binary' and dimension==10")
display(primary)
display(metrics.query("method=='centroid_direction' and dimension==10").sort_values('auc',ascending=False))

## 5. Why centroid direction and signed cosine are related

With row-normalized embeddings, projection on the unnormalized centroid difference equals mean cosine similarity to positive firms minus mean cosine similarity to negative firms. Normalizing the direction and applying the centroid midpoint changes calibration and can slightly change leave-one-out ranks, but it does not create new embedding information. Their empirical correlation is reported explicitly.

In [ ]:
equivalence=(predictions.groupby(['representation','dimension'])
 .apply(lambda g:g.centroid_margin.corr(g.class_balanced_cosine,method='spearman'),include_groups=False)
 .rename('spearman_centroid_vs_cosine').reset_index())
equivalence.to_parquet(RESULTS/'centroid_cosine_equivalence.parquet',index=False)
display(equivalence.query('dimension==10'))

## 6. Frozen permutation inference

Labels are permuted while preserving the observed 29/55 class counts. Every permutation reconstructs all leave-one-out centroids. The primary p-value asks how often a random labeling produces at least as much AUC as the observed labeling.

In [ ]:
permutation_summaries=[];null_frames=[];started=time.perf_counter()
for offset,(name,frame) in enumerate(representations.items()):
    summary,null=dc.permutation_test(frame,labels,name,dimension=10,repetitions=5000,seed=20220224+offset)
    permutation_summaries.append(summary);null_frames.append(null);print(name,'complete')
permutation_summary=pd.concat(permutation_summaries,ignore_index=True)
permutation_null=pd.concat(null_frames,ignore_index=True)
permutation_summary['seconds_total']=time.perf_counter()-started
permutation_summary.to_parquet(RESULTS/'permutation_summary_10d.parquet',index=False)
permutation_null.to_parquet(RESULTS/'permutation_null_10d.parquet',index=False)
display(permutation_summary.sort_values('observed_auc',ascending=False))

## 7. Dimension sensitivity

We report every pre-specified dimension rather than choosing the best result. Lower-dimensional variants retain the first components of the existing ordered 10D representation and are renormalized.

In [ ]:
plot=metrics.query("method=='centroid_direction'").pivot(index='dimension',columns='representation',values='auc')
ax=plot.plot(marker='o',figsize=(11,6));ax.axhline(.5,color='black',ls='--',lw=1);ax.set_ylabel('Leave-one-out AUC');ax.set_ylim(0,1);ax.legend(bbox_to_anchor=(1.02,1),loc='upper left');plt.tight_layout();plt.savefig(RESULTS/'auc_by_dimension.png',dpi=180,bbox_inches='tight');plt.show()
display(plot)

## 8. Characteristics and industry baselines

The characteristics benchmark receives the same centroid procedure. The industry-only score is the leave-one-out positive rate within SIC2, falling back to the global training rate when a cell is too small.

In [ ]:
reference=representations['all_s12_rs_binary'][['permco','firm_name']]
characteristics=dc.characteristic_frame(ROOT,reference)
char_columns=[c for c in characteristics if c not in ['permco','firm_name']]
characteristics=characteristics.rename(columns={c:f'embedding_{i:03d}' for i,c in enumerate(char_columns,1)})
char_metrics,char_predictions=dc.evaluate_representation(characteristics,labels,'characteristics',dimensions=(len(char_columns),))
industry=pd.read_parquet(GEO/'event_data/crsp_point_in_time_industry_2021q4.parquet')
industry_predictions,industry_metrics=dc.industry_loo_scores(labels,industry)
baseline=pd.concat([char_metrics,pd.DataFrame([{'representation':'industry_only','dimension':0,'method':'loo_sic2_rate','firms':84,**industry_metrics}])],ignore_index=True)
baseline.to_parquet(RESULTS/'characteristic_industry_baselines.parquet',index=False)
industry_predictions.to_parquet(RESULTS/'industry_loo_predictions.parquet',index=False)
display(baseline)

## 9. Industry-residualized embedding test

Each embedding coordinate is demeaned within SIC2 using the full pre-event embedding universe—without labels or returns—before repeating leave-one-out centroid validation. This asks whether polarity survives removal of broad industry location.

In [ ]:
residual_rows=[]
for name,frame in representations.items():
    residual=dc.residualize_industry(frame,industry,10)
    result,_=dc.evaluate_representation(residual,labels,name+'_industry_residual',dimensions=(10,))
    residual_rows.append(result)
residual_metrics=pd.concat(residual_rows,ignore_index=True)
residual_metrics.to_parquet(RESULTS/'industry_residualized_metrics.parquet',index=False)
display(residual_metrics.query("method=='centroid_direction'").sort_values('auc',ascending=False))

## 10. Mixed-firm diagnostic

The five mixed firms never train the direction. Their full-direction margins are descriptive: placement near zero would be consistent with offsetting mechanisms, but five observations cannot provide a formal test.

In [ ]:
mixed=[]
for name,frame in representations.items():
    result=dc.mixed_firm_scores(frame,labels,10);result['representation']=name;mixed.append(result)
mixed=pd.concat(mixed,ignore_index=True);mixed.to_parquet(RESULTS/'mixed_firm_diagnostics.parquet',index=False)
display(mixed.sort_values(['firm_id','representation']))

## 11. Outcome-free decision gate

Proceed to universe scoring and exploratory Ukraine returns only if the directional model shows economically meaningful separation—approximately AUC ≥ 0.65—with permutation support, and preferably retains incremental structure after industry removal. If performance is near chance, stop. If raw performance disappears after residualization, interpret the direction as primarily sectoral. No return result may revise this gate.

In [ ]:
primary_row=permutation_summary.query("representation=='all_s12_rs_binary'").iloc[0]
primary_residual=residual_metrics.query("representation=='all_s12_rs_binary_industry_residual' and method=='centroid_direction'").iloc[0]
decision='proceed' if primary_row.observed_auc>=.65 and primary_row.auc_permutation_p<=.05 else 'stop_or_reformulate'
gate=pd.DataFrame([{'primary_auc':primary_row.observed_auc,'primary_permutation_p':primary_row.auc_permutation_p,'industry_residual_auc':primary_residual.auc,'threshold_auc':.65,'threshold_p':.05,'decision':decision,'event_returns_loaded':False}])
gate.to_parquet(RESULTS/'outcome_free_decision_gate.parquet',index=False)
display(gate)